# Bài 1 v4 · Vector và độ tương đồng cosine



In [6]:
from pathlib import Path
import sys, math
import numpy as np

ROOT = Path.cwd()
if not (ROOT / "source/cosine_lab.py").exists():
    raise RuntimeError("Mở notebook tại thư mục Bai01_v4 chứa source/ và data/.")
sys.path.insert(0, str(ROOT / "source"))
from cosine_lab import load_movies, labels, encode, cosine_safe, AXES3, QUERY, CANDIDATES
from sklearn.metrics.pairwise import cosine_similarity

# Chọn phim tham chiếu và 2 phim ứng viên mới
QUERY = 1270
CANDIDATES = (480,1)

movies = load_movies()
print("Số phim:", len(movies))
print("Ba trục:", AXES3)
print("Dòng nguồn:", movies[QUERY])


Số phim: 9742
Ba trục: ('Action', 'Comedy', 'Sci-Fi')
Dòng nguồn: {'movieId': '1270', 'title': 'Back to the Future (1985)', 'genres': 'Adventure|Comedy|Sci-Fi'}


## 1. Dữ liệu → vector


In [8]:
u = encode(movies[QUERY], AXES3)
X = np.stack([
    encode(movies[mid], AXES3)
    for mid in CANDIDATES
    ])
print("u =", u.astype(int))
print("X (2 × 3) =\n", X.astype(int))
for mid, v in zip(CANDIDATES, X):
    print(mid, movies[mid]["title"], "→", v.astype(int))
assert X.shape == (2, 3)


u = [0 1 1]
X (2 × 3) =
 [[1 0 1]
 [0 1 0]]
480 Jurassic Park (1993) → [1 0 1]
1 Toy Story (1995) → [0 1 0]


## 2. Một hàng tính có bước


In [9]:
# Lấy vecto Jurassic Park
v = X[0]
#  Tích vô hướng
dot = float(u @ v)
#  Độ dài 2 vecto
nu, nv = float(np.linalg.norm(u)), float(np.linalg.norm(v))
print("dot =", dot, "; chuẩn u =", nu, "; chuẩn v =", nv)
# Tính cosine
hand = dot / math.sqrt(float(u @ u) * float(v @ v))
# So sánh với hàm của bài và scikit-learn
own = cosine_safe(u, v)
lib = float(cosine_similarity([u], [v])[0, 0])
print("Tính tay / NumPy / scikit-learn:", hand, own, lib)
assert abs(hand - own) < 1e-12 and abs(own - lib) < 1e-12
for mid, v in zip(CANDIDATES, X):
    print(
        movies[mid]["title"],
        "dot =", int(u @ v),
        "cos =", format(cosine_safe(u, v), ".4f")
    )

dot = 1.0 ; chuẩn u = 1.4142135623730951 ; chuẩn v = 1.4142135623730951
Tính tay / NumPy / scikit-learn: 0.5 0.4999999999999999 0.4999999999999999
Jurassic Park (1993) dot = 1 cos = 0.5000
Toy Story (1995) dot = 1 cos = 0.7071


## 3. Điều kiện và phản ví dụ


In [10]:
# Toy Story theo 3 trục đã chọn
toy_vector = encode(movies[1],AXES3)
print("Toy Story:", cosine_safe(u, toy_vector))
# Vector 0 : không có thể loại nào trên 3 trục
zero = np.zeros(3)
print("Vector 0:", cosine_safe(u, zero))
# So sánh với quy ước của scikit-learn
print("Quy ước thư viện:", cosine_similarity([u], [zero])[0, 0])
# Kiểm tra tính chất
assert abs(cosine_safe(u, toy_vector) - 1 / np.sqrt(2)) < 1e-12
assert cosine_safe(u, zero) is None
assert abs(cosine_safe(3 * u, X[0]) - cosine_safe(u, X[0])) < 1e-12
# Các trường hợp minh họa
for a, b in [([1, 0], [0, 1]), ([1, 0, 1], [0, 1, 1]), ([1], [1]), ([1, 0], [1, 1])]:
    print(a, b, "→", cosine_safe(a, b))


Toy Story: 0.7071067811865475
Vector 0: None
Quy ước thư viện: 0.0
[1, 0] [0, 1] → 0.0
[1, 0, 1] [0, 1, 1] → 0.4999999999999999
[1] [1] → 1.0
[1, 0] [1, 1] → 0.7071067811865475


## 4. Giữ phim, thay biểu diễn


In [16]:
import pandas as pd
# So sánh cosine trên 3 nhãn và 19 nhãn
axes_full = sorted(set().union(*(labels(r) for r in movies.values())))
u_full = encode(movies[QUERY], axes_full)
a = labels(movies[QUERY])

for mid in CANDIDATES:
    b = labels(movies[mid])

    cos3 = cosine_safe(u, encode(movies[mid], AXES3))
    cos19 = cosine_safe(u_full, encode(movies[mid], axes_full))

    assert abs(cos19 - len(a & b) / math.sqrt(len(a) * len(b))) < 1e-12

    print(
        movies[mid]["title"],
        "-> cosine 3 nhãn:", round(cos3, 4),
        "| cosine 19 nhãn:", round(cos19, 4)
    )



Jurassic Park (1993) -> cosine 3 nhãn: 0.5 | cosine 19 nhãn: 0.5774
Toy Story (1995) -> cosine 3 nhãn: 0.7071 | cosine 19 nhãn: 0.5164


## 5. Vận dụng và tự chấm


In [17]:
v = np.array([0., 1., 1.])
print("dot", u @ v, "; chuẩn", np.linalg.norm(u), np.linalg.norm(v))
print("cos", cosine_safe(u, v), "; 3u", cosine_safe(3*u, v), "; zero", cosine_safe(u, np.zeros(3)))
assert abs(cosine_safe(u, v) - 1.0) < 1e-12
assert abs(cosine_safe(3*u, v) - 1.0) < 1e-12
assert cosine_safe(u, np.zeros(3)) is None
from cosine_lab import run
report = run()
print("Kiểm toàn kho:", report["candidate_pool"], "ứng viên;", report["exact_equal_label_count"], "phim khác cùng tập nhãn")


dot 2.0 ; chuẩn 1.4142135623730951 1.4142135623730951
cos 0.9999999999999998 ; 3u 1.0 ; zero None
{"status": "pass", "checks": 38, "candidate_pool": 9707, "maximum_error": 2.220446049250313e-16}
Kiểm toàn kho: 9707 ứng viên; 42 phim khác cùng tập nhãn


 **Nhận xét**
Với 3 nhãn thể loại Action, Comedy và Sci-Fi , Toy Story có độ tương đồng cosine cao hơn Back to the Futre(0,7071) so với Jurassic Park(0,5000). Tuy nhiên , khi mở rộng lên 19 nhãn thể loại, kết quả thay đổi: Jurassic Park đạt 0,5774, cao hơn Toy Story(0,5164). Điều này cho thấy việc lựa chọn các trục biểu diễn ảnh hưởng đến kết quả so sánh . Cosine phản ánh mức độ tương đồng giữa các vector thể loại nhưng không trực tiếp thể hiện chất lượng phim hay sở thích của người xem . Vì vậy , không nên chỉ dựa vào chỉ số này để kết luận phim nào phù hợp hơn.


